# Translation from English to Spanish using Flan-T5 and Helsinki-NLP/opus-100 Dataset


## Introduction
In this notebook, we will use the Flan-T5 model to perform translation from English to Spanish. We will use the "Helsinki-NLP/opus-100" dataset from Hugging Face, specifically the en-es subset, to train and evaluate our translation model.


In [1]:
!pip install transformers tensorflow datasets

In [2]:

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## Loading the Dataset

In [3]:
#Loads the English–Spanish portion of the OPUS-100 translation dataset.
from datasets import load_dataset

# Load the Helsinki-NLP/opus-100 dataset
dataset = load_dataset('Helsinki-NLP/opus-100', 'en-es')
print(dataset['train'][0])


README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

en-es/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  237kB            

en-es/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en-es/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 99.6MB            

en-es/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

en-es/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'translation': {'en': "It was the asbestos in here, that's what did it!", 'es': 'Fueron los asbestos aquí. ¡Eso es lo que ocurrió!'}}


## Data Preprocessing

In [4]:
#Defines a function for preparing several translation examples at once.
def preprocess_data(examples):
    inputs = [
        f"Translate from English to Spanish: {x['en']}"
        for x in examples["translation"]
    ]
    targets = [
        x["es"]
        for x in examples["translation"]
    ]

    return tokenizer(
        inputs,
        text_target=targets,
        max_length=128,
        truncation=True
        # Do not use padding="max_length"
    )



## Converting to TensorFlow Datasets

In [5]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq
) #Imports classes for loading the tokenizer, loading the model, and preparing batches.
from torch.utils.data import DataLoader #Imports PyTorch’s DataLoader for creating batches of data.


train_size = min(30000, len(dataset["train"])) #Chooses up to 30,000 training examples without exceeding the available dataset size.

processed_train_dataset = (
    dataset["train"]
    .select(range(train_size))
    .map(
        preprocess_data,
        batched=True,
        remove_columns=dataset["train"].column_names
    )
) #Selects the first 30,000 training examples and preprocesses them in batches.

processed_test_dataset = dataset["test"].map(
    preprocess_data,
    batched=True,
    remove_columns=dataset["test"].column_names
)  #Preprocesses all examples in the test dataset.

processed_validation_dataset = dataset["validation"].map(
    preprocess_data,
    batched=True,
    remove_columns=dataset["validation"].column_names
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    return_tensors="pt"
) #Creates a collator that combines individual examples into padded PyTorch batches suitable for the model.


train_dataloader = DataLoader(
    processed_train_dataset,
    shuffle=True,
    batch_size=8,
    collate_fn=data_collator
) #Creates training batches of eight examples and randomly changes their order.

test_dataloader = DataLoader(
    processed_test_dataset,
    shuffle=False,
    batch_size=8,
    collate_fn=data_collator
) #Creates test batches of eight examples without changing their order.


batch = next(iter(train_dataloader)) #Retrieves the first batch from the training data loader.

print(batch.keys()) #Prints the available batch fields, normally input_ids, attention_mask, and labels.

for key, value in batch.items():
    print(key, value.shape) #Prints the shape of every tensor in the batch.

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

KeysView({'input_ids': tensor([[30355,    15,    45,  1566,    12,  5093,    10,     3,  7371,     6,
            62,    33,   230,  8154,     3,     9,  4431,   483,    16,   994,
           855,    75,    17,  1628,    81,  1043,  1596,    10,    59,   163,
            33,   469,    22,     7,  1596,   882,  6739,    16,   490,  1353,
            57,  4332,  2443,     6,    68,     8,   647,     7,  3212,  8482,
          8367,    24,  1596,    33,  1644,    12,  2367,   306,    21,   203,
            12,   287,    15,     5,     1],
        [30355,    15,    45,  1566,    12,  5093,    10,   264,   959,  1307,
            25,   241,    12,   103,  8988,    58,     1,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0

## Freezing the Model

In [6]:
print(model.config) #Prints the model’s complete configuration.

T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2048,
  "d_kv": 64,
  "d_model": 768,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "dtype": "float32",
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 12,
  "num_heads": 12,
  "num_layers": 12,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "scale_decoder_outputs": false,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {


In [7]:
print("Model type:", model.config.model_type) #Prints the model architecture type, such as t5.
print("Hidden dimension:", model.config.d_model) #Prints the size of the hidden representation used inside the model.
print("Encoder/decoder layers:", model.config.num_layers) #Prints the configured number of transformer layers.
print("Attention heads:", model.config.num_heads) #Prints the number of attention heads in each attention layer.
print("Vocabulary size:", model.config.vocab_size) #Prints the number of tokens known by the model.

Model type: t5
Hidden dimension: 768
Encoder/decoder layers: 12
Attention heads: 12
Vocabulary size: 32128


In [8]:
# model.shared.requires_grad_(False) #Freezes the shared token-embedding layer so that its weights are not updated.
model.encoder.requires_grad_(False) #Freezes the entire encoder.
# model.decoder.requires_grad_(False) #Freezes the entire decoder.

T5Stack(
  (embed_tokens): Embedding(32128, 768)
  (block): ModuleList(
    (0): T5Block(
      (layer): ModuleList(
        (0): T5LayerSelfAttention(
          (SelfAttention): T5Attention(
            (q): Linear(in_features=768, out_features=768, bias=False)
            (k): Linear(in_features=768, out_features=768, bias=False)
            (v): Linear(in_features=768, out_features=768, bias=False)
            (o): Linear(in_features=768, out_features=768, bias=False)
            (relative_attention_bias): Embedding(32, 12)
          )
          (layer_norm): T5LayerNorm()
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (1): T5LayerFF(
          (DenseReluDense): T5DenseGatedActDense(
            (wi_0): Linear(in_features=768, out_features=2048, bias=False)
            (wi_1): Linear(in_features=768, out_features=2048, bias=False)
            (wo): Linear(in_features=2048, out_features=768, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
      


### Important Considerations in Transfer Learning

1. **Freezing the LLM Layer:** In transfer learning, it's important to freeze the pre-trained language model layer to retain the knowledge it has already acquired and to avoid overfitting. This allows the model to leverage its pre-trained capabilities while focusing on learning the new task-specific nuances.

2. **Loss Function with `from_logits=True`:** When fine-tuning language models from Hugging Face, it's crucial to use the loss function with `from_logits=True`. This is because these models do not apply softmax to their outputs, and using `from_logits=True` ensures that the loss is computed correctly.


## Model Training

In [9]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 113.0 MB/s eta 0:00:00


In [10]:
import torch
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
# Imports the trainer, its configuration class, and the sequence-to-sequence data collator.
# -------------------------------------------------
# 1. Optional: freeze selected parts of the model
# -------------------------------------------------

# Example: freeze only the encoder
model.encoder.requires_grad_(False)

# Do not freeze encoder, decoder, and shared embeddings together,
# because that would leave almost nothing useful to train.


# -------------------------------------------------
# 2. Create the data collator
# -------------------------------------------------

# data_collator = DataCollatorForSeq2Seq(
#     tokenizer=tokenizer,
#     model=model,
#     padding=True,
#     return_tensors="pt"
# )  #already run before hence commenting


# -------------------------------------------------
# 3. Define training settings
# -------------------------------------------------

training_args = Seq2SeqTrainingArguments( #Begins defining the model-training settings.
    output_dir="./flan_t5_translation", #Sets the folder where checkpoints and training outputs will be stored.

    # Training
    num_train_epochs=3, #Trains the model for three complete passes through the training dataset.
    learning_rate=5e-5, #Sets the rate at which the model’s trainable weights are updated.
    per_device_train_batch_size=8, #Uses eight training examples per batch on each device.
    per_device_eval_batch_size=8, #Uses eight validation examples per batch on each device.

    # Validation
    eval_strategy="epoch", #Evaluates the model after every epoch.

    # Save the model after every epoch
    save_strategy="epoch",
    save_total_limit=2, #Keeps only the two most recent or most relevant checkpoints.

    # Logging
    logging_strategy="steps", #Records training information after a specified number of steps.
    logging_steps=100,

    # Generation during evaluation
    predict_with_generate=True, #Makes the model generate translations during evaluation.
    generation_max_length=128, #Limits generated evaluation sequences to 128 tokens.

    # Use GPU mixed precision when available
    # fp16=torch.cuda.is_available(), #Uses faster 16-bit calculations when a compatible GPU is available. but caused training and validation loss nan issue
    fp16=False,
    bf16=False,
    logging_nan_inf_filter=False,  # expose the real problem

    # Prevent external logging services from being started
    report_to="none", #Prevents the trainer from sending results to external logging services.

    # Keep the best checkpoint
    load_best_model_at_end=True, #Restores the best saved checkpoint after training finishes.
    metric_for_best_model="eval_loss", #Uses validation loss to decide which checkpoint is best.
    greater_is_better=False #States that a lower validation loss represents a better model.
)


# -------------------------------------------------
# 4. Create the trainer
# -------------------------------------------------

trainer = Seq2SeqTrainer( # Creates the Hugging Face object that manages training and evaluation.
    model=model, # Provides the FLAN-T5 model that will be trained.
    args=training_args, # Provides the previously defined training settings.
    train_dataset=processed_train_dataset, # Uses the processed training dataset for training.
    eval_dataset=processed_validation_dataset, # Uses the processed test dataset for evaluation.
    data_collator=data_collator,# Uses the collator to prepare each batch.
    processing_class=tokenizer # Provides the tokenizer for processing and saving purposes.
)


# -------------------------------------------------
# 5. Train the model
# -------------------------------------------------

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.598441,1.309452
2,1.512020,1.284377
3,1.471141,1.280684


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=11250, training_loss=1.526442142062717, metrics={'train_runtime': 1163.4234, 'train_samples_per_second': 77.358, 'train_steps_per_second': 9.67, 'total_flos': 6074428976898048.0, 'train_loss': 1.526442142062717, 'epoch': 3.0})

## Performing Translation

In [11]:
import torch

# Select GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #Chooses a GPU when available; otherwise, it chooses the CPU.
model.to(device) #Moves the model to the selected device.

# Set model to evaluation mode
model.eval() #Places the model in evaluation mode so its weights are not treated as being under training.


def translate(example): #Defines a function that translates one processed example.
    # Convert the stored token IDs into a PyTorch tensor
    input_ids = torch.tensor(
        example["input_ids"],
        dtype=torch.long
    ).unsqueeze(0).to(device) #Converts the stored input token IDs into a PyTorch tensor, adds a batch dimension, and moves it to the selected device.

    attention_mask = torch.tensor(
        example["attention_mask"],
        dtype=torch.long
    ).unsqueeze(0).to(device) #Converts the attention mask into a batched PyTorch tensor and moves it to the same device.

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=128,
            num_beams=4,
            early_stopping=True
        ) #Disables gradient calculations because the model is only generating translations.
        #Generates a translation using beam search with four candidate paths.
        #The tokenization, generation, and beam-search pattern is a repeat of Codes 1 and 2, but it is now applied to a fine-tuned model and dataset example.

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ) #Converts the generated token IDs into readable text and returns it.


# Translate five examples
number_of_examples = min(5, len(processed_test_dataset)) #Chooses up to five test examples.

for i in range(number_of_examples): #Repeats the following operations for each selected example.
    example = processed_test_dataset[i] #Retrieves one processed test example.

    translated_text = translate(example) #Uses the translation function to generate its Spanish translation.

    # Decode the English input
    input_text = tokenizer.decode(
        example["input_ids"],
        skip_special_tokens=True
    ) #Converts the English input token IDs back into readable text.

    # Labels may contain -100, which must be replaced before decoding #Replaces ignored label values (-100) with padding tokens so the reference translation can be decoded.
    label_ids = [
        token_id if token_id != -100 else tokenizer.pad_token_id
        for token_id in example["labels"]
    ]

    reference_text = tokenizer.decode(
        label_ids,
        skip_special_tokens=True
    ) #Converts the correct Spanish label tokens into readable text.

    print(f"Input: {input_text}")
    print(f"Reference Translation: {reference_text}")
    print(f"Translated Text: {translated_text}")
    print("-" * 80)

Input: Translate from English to Spanish: If your country produced ODS for this purpose, please enter the amount so produced in column 6 on Data Form 3.”
Reference Translation: Si su pas produjo SAO para estos usos, srvase anotar en la columna 6 del formulario de datos 3 la cantidad correspondiente”.
Translated Text: Si su pas produció ODS para esta propósito, entiendo la cantidad que produció en la columna 6 en el formulario de datos 3”.
--------------------------------------------------------------------------------
Input: Translate from English to Spanish:  juvie the great man, who else could it be but I?
Reference Translation: # Juvie el gran hombre, quién podra ser sino yo?
Translated Text:  Juvie el gran hombre, qué más podra ser sino yo?
--------------------------------------------------------------------------------
Input: Translate from English to Spanish: The home planet is running out.
Reference Translation: El planeta madre se está agotando.
Translated Text: El planeta domi


## Conclusion
In this notebook, we used the Flan-T5 model to perform translation from English to Spanish using the Helsinki-NLP/opus-100 dataset. Preprocessed the dataset, fine-tuned the model while freezing the LLM layer, and performed translations. We manually validated the translations to assess the quality of the model's performance. The results demonstrate the effectiveness of the Flan-T5 model for translation tasks.
